# Fine-tuning del extractor semántico (Colab T4)

Este notebook es un **envoltorio delgado** del motor de entrenamiento ya validado
(`entrenamiento/qlora_entrenar.py`): una sola fuente de verdad, sin duplicar código.

**Qué hace el motor** (todo en un comando):
- Carga `train.jsonl` (3200) / `dev.jsonl` (200) / `hard_dev.jsonl` (150)
- Formato **prompt-completación** con `completion_only_loss=True`: la loss va SOLO
  sobre el JSON objetivo, no sobre el prompt
- QLoRA 4-bit NF4 + LoRA (fp16 en T4), selecciona checkpoint por `eval_loss`
- Al terminar evalúa **exact-match** campo a campo sobre `hard_dev` y `dev`
- Guarda adapter + `config_entrenamiento.json`

**Regla de no fuga**: solo entrena con datos sintéticos; Benchmark2 es solo test.

**Orden**: (1) instalar → (2) subir proyecto → (3) smoke test → (4) entrenar → (5) publicar.

In [ ]:
# 1) GPU y dependencias
# Versiones fijadas = las mismas con las que se valido el motor (smoke test local).
# bitsandbytes no se puede validar sin CUDA: va la ultima.
# torchao: Colab trae 0.10 y peft 0.21 aborta si encuentra una version < 0.16; no se usa.
!nvidia-smi
%pip install -q "transformers==5.17.0" "trl==0.21.0" "peft==0.21.0" "datasets==5.0.1" "accelerate==1.15.0" bitsandbytes
%pip uninstall -y -q torchao

In [ ]:
# 2) Traer el proyecto (sube proyecto.zip, generado con: python datos/preparar_zip.py)
import os, sys, shutil, zipfile

if not os.path.isfile(os.path.join("proyecto", "entrenamiento", "qlora_entrenar.py")):
    if os.path.isdir("proyecto"):
        shutil.rmtree("proyecto")
    from google.colab import files
    print("Sube proyecto.zip (raiz del proyecto comprimida):")
    subido = files.upload()
    zip_path = next(iter(subido))
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            nombre = info.filename.replace("\\", "/")
            if not nombre or nombre.endswith("/") or "__pycache__" in nombre:
                continue
            ruta = os.path.join("proyecto", *nombre.split("/"))
            os.makedirs(os.path.dirname(ruta), exist_ok=True)
            with z.open(info) as src, open(ruta, "wb") as dst:
                shutil.copyfileobj(src, dst)
    # si el zip tenia una carpeta raiz unica, la bajamos un nivel
    hijos = [h for h in os.listdir("proyecto") if h != "__MACOSX"]
    if len(hijos) == 1 and os.path.isdir(os.path.join("proyecto", hijos[0])):
        raiz_zip = os.path.join("proyecto", hijos[0])
        for h in os.listdir(raiz_zip):
            shutil.move(os.path.join(raiz_zip, h), os.path.join("proyecto", h))
        os.rmdir(raiz_zip)

RAIZ = os.path.abspath("proyecto")
sys.path.insert(0, RAIZ)
os.chdir(RAIZ)          # el motor trabaja con rutas relativas al proyecto
print("Proyecto en:", RAIZ)
!python entrenamiento/qlora_entrenar.py --help

## 3) Smoke test (~4 min) — NO saltar

Corre el motor 20 pasos con un modelo diminuto aleatorio: valida el código completo
(datos, LoRA, bucle, guardado) **antes** de gastar tiempo en el modelo real. Debe
terminar con "adapter + configuracion guardados".

In [ ]:
# 3) Smoke test del codigo (el modelo es aleatorio: NO es evidencia)
!python entrenamiento/qlora_entrenar.py --smoke --max-steps 20 --epocas 1 --salida /tmp/smoke_adapter

## 4) Entrenamiento real (T4)

`Qwen3-4B-Instruct-2507` con LoRA r=16 sobre los 7 módulos, 3 épocas. Al terminar el
motor imprime el **exact-match de `hard_dev`** (patrones difíciles: artículos de
participantes, títulos derivados) y de `dev` (general), campo a campo, con los fallos.

**Antes de que empiece a entrenar, el motor verifica:**
- `tokens por ejemplo`: el prompt de reglas ocupa ~520 tokens y cada ejemplo 519–591;
  `max_length` se calcula de los datos (nunca se trunca el JSON objetivo).
- `mascara de loss verificada`: solo ~43–66 tokens por ejemplo (el JSON) reciben loss.

**Duración**: cada ejemplo tiene ~560 tokens, así que el costo lo domina el prompt. Mira
el ETA de la barra de progreso en los primeros pasos. Los checkpoints van a **Drive**
cada 100 pasos: si Colab se desconecta, re-ejecuta la celda con `--reanudar` y continúa.

**Criterio de éxito**: `participantes` ≈ 100% en `hard_dev`. Esa es la convención que
hoy falla 6 veces en Benchmark2.

In [ ]:
# 4) Entrenamiento real -- checkpoints en Drive para sobrevivir desconexiones
from google.colab import drive
drive.mount('/content/drive')
SALIDA = "/content/drive/MyDrive/qwen3-4b-semantic-lora"

# primera vez: sin --reanudar. Si Colab se corta, vuelve a correr esta celda
# con REANUDAR = True y sigue desde el ultimo checkpoint guardado en Drive.
REANUDAR = False
!python entrenamiento/qlora_entrenar.py \
    --modelo Qwen/Qwen3-4B-Instruct-2507 \
    --epocas 3 \
    --salida {SALIDA} {"--reanudar" if REANUDAR else ""}

## 5) (Opcional) Experimento de tamaño — rubro *Model economy*

El rubro da nota completa al **modelo más chico que funcione**. Con el mismo motor:

```python
# 0.5B (rápido: ~10 min)          # 1.7B
!python entrenamiento/qlora_entrenar.py --modelo Qwen/Qwen2.5-0.5B-Instruct --epocas 6 --salida adapter-0.5b
!python entrenamiento/qlora_entrenar.py --modelo Qwen/Qwen3-1.7B --epocas 4 --salida adapter-1.7b
```

Registra el exact-match de cada uno y quédate con el más chico que cumpla la barra
(brazo D ≥ 95% en Benchmark2). Si usas un modelo fuera de los 3 candidatos de D1,
la desviación se declara y defiende en el documento (texto listo en el README).

In [ ]:
# 5) (Opcional) descomenta el tamano a probar
# !python entrenamiento/qlora_entrenar.py --modelo Qwen/Qwen2.5-0.5B-Instruct --epocas 6 --salida adapter-0.5b
# !python entrenamiento/qlora_entrenar.py --modelo Qwen/Qwen3-1.7B --epocas 4 --salida adapter-1.7b

In [ ]:
# 6) Publicar el adapter (respaldo permanente)
# a) ya esta en Drive: /content/drive/MyDrive/qwen3-4b-semantic-lora
#    (adapter final + config_entrenamiento.json con el exact-match de hard_dev/dev)
!ls -la /content/drive/MyDrive/qwen3-4b-semantic-lora
!cat /content/drive/MyDrive/qwen3-4b-semantic-lora/config_entrenamiento.json | head -60

# b) subelo a HF Hub (necesita token: huggingface.co/settings/tokens)
# from huggingface_hub import notebook_login
# notebook_login()
# from huggingface_hub import HfApi
# api = HfApi()
# api.create_repo("tu-usuario/qwen3-4b-semantic-lora", exist_ok=True)
# api.upload_folder(folder_path="/content/drive/MyDrive/qwen3-4b-semantic-lora",
#                   repo_id="tu-usuario/qwen3-4b-semantic-lora",
#                   ignore_patterns=["checkpoint-*"])

In [ ]:
# 7) (Referencia) usar el adapter en el brazo D del notebook principal
# en notebooks/deliverable2_colab.ipynb:
#   ADAPTER = "/content/drive/MyDrive/qwen3-4b-semantic-lora"
#   ADAPTER = "tu-usuario/qwen3-4b-semantic-lora"   # desde HF Hub
# luego correr los brazos A-D con --adapter

## Qué debe salir de esta sesión

1. `qwen3-4b-semantic-lora/` (adapter + `config_entrenamiento.json`) — en Drive y/o HF Hub
2. Exact-match de `hard_dev` y `dev` impresos por el motor (anótalos para el documento)
3. Opcional: tabla de tamaños (0.5B / 1.7B / 4B) para el rubro *Model economy*

**Siguiente paso**: `notebooks/deliverable2_colab.ipynb` → `ADAPTER = <ruta>` → re-correr
el brazo D sobre las 45 frases → comparar contra el 84.4% del modelo base.